In [1]:
import itertools
import numpy as np
import gurobipy as gp
from gurobipy import GRB




In [2]:
def optimal_min_cut_balanced(n, edges,weights):
    assert len(weights) == len(edges)
    model = gp.Model()
    model.Params.OutputFlag = 0

    # x_i in {0,1}
    x = model.addVars(n, vtype=GRB.BINARY, name="x")

    # c_e = 1 if hyperedge e is cut
    c = model.addVars(len(edges), vtype=GRB.BINARY, name="cut")

    target_ones = (n + 1) // 2
    # exact balance: half the vertices in partition 1
    model.addConstr(gp.quicksum(x[i] for i in range(n)) ==target_ones,name="balance")

    # AoN cut constraints
    for e_idx, edge in enumerate(edges):
        for u, v in itertools.combinations(edge, 2):
            #enforce that the edge is cut if vertexes in different partitions
            #essentially implementation of max |x_u-x_v| ==L(x) so cost must be 1 or 0
            model.addConstr(c[e_idx] >= x[u] - x[v])
            model.addConstr(c[e_idx] >= x[v] - x[u])

    # minimize total number of cut hyperedges
    objective = gp.quicksum(weights[e_idx]*c[e_idx] for e_idx in range(len(edges)))

    model.setObjective(objective, GRB.MINIMIZE)

    model.optimize()

    assert model.Status == GRB.OPTIMAL, f"Gurobi did not prove optimality. Status = {model.Status}"
    # print(x)
    # print(c)
    x_opt = np.array([int(round(x[i].X))for i in range(n)])
    cut_opt = np.array([int(round(c[e_idx].X)) for e_idx in range(len(edges))])


    return x_opt, cut_opt, model.ObjVal

In [3]:
n = 6

edges = [
    (0, 1, 2),
    (3, 4, 5),
    (0, 3)
]
weights = [1,1,1]
mini_example=[n,edges,weights]

In [4]:
x_opt, cut_opt, obj = optimal_min_cut_balanced(mini_example[0], mini_example[1],mini_example[2])

print("Optimal partition:", x_opt)
print("Cut indicators:", cut_opt)
print("Optimal cost:", obj)

Restricted license - for non-production use only - expires 2027-11-29
Optimal partition: [0 0 0 1 1 1]
Cut indicators: [0 0 1]
Optimal cost: 1.0


In [5]:
def get_edge_list(P):
    edges=[]
    for idx in range(P.shape[1]):
        #indices that are not 0
        edge = tuple(np.flatnonzero(P[:,idx]))
        edges.append(edge)
    return edges

In [6]:
import os
os.getcwd()

'/Users/stevehalley/Code/Github/QuantumHypergraphPartitioning'

In [12]:
import os
import glob
import numpy as np


poisson_types = [
    "poisson_16_12_3",
    "poisson_16_12_4",
    "poisson_16_12_5"
]

poisson_results = []

for graph_type in poisson_types:

    p_folder = f"data/{graph_type}/P"
    w_folder = f"data/{graph_type}/w"

    p_files = sorted(glob.glob(f"{p_folder}/*.npy"))

    for p_file in p_files:
        instance = os.path.splitext(os.path.basename(p_file))[0]

        # Load stochastic incidence P

        P = np.load(p_file)
        n = P.shape[0]
        edges = get_edge_list(P)

        w_file = os.path.join(w_folder,f"{instance}.npy")
        weights = np.load(w_file)

        # make sure weights are 1D
        weights = np.asarray(weights).reshape(-1)

        assert len(weights) == len(edges), (
            f"{graph_type}/{instance}: "
            f"{len(weights)} weights different than "
            f"{len(edges)} edges"
        )

        x_opt, cut_opt, optimal_cost = (ptimal_min_cut_balanced(n,edges,weights))

        # optional: unweighted number of cut edges
        num_cut = int(cut_opt.sum())

        poisson_results.append({
            "dataset": graph_type,
            "instance": instance,
            "n_vertices": n,
            "n_edges": len(edges),
            "optimal_partition":
                "".join(map(str, x_opt)),
            "optimal_weighted_cost":
                optimal_cost,
            "num_cut_edges":
                num_cut,
            "cut_indicators":
                "".join(map(str, cut_opt))
        })

        print(
            graph_type,
            instance,
            "weighted cost =",optimal_cost,
            "| cut edges =",num_cut,
            
        )

poisson_16_12_3 01 weighted cost = 6.0 | cut edges = 6
poisson_16_12_3 02 weighted cost = 6.0 | cut edges = 6
poisson_16_12_3 03 weighted cost = 8.0 | cut edges = 8
poisson_16_12_3 04 weighted cost = 10.0 | cut edges = 10
poisson_16_12_3 05 weighted cost = 7.0 | cut edges = 7
poisson_16_12_3 06 weighted cost = 10.0 | cut edges = 10
poisson_16_12_3 07 weighted cost = 8.0 | cut edges = 8
poisson_16_12_3 08 weighted cost = 7.0 | cut edges = 7
poisson_16_12_3 09 weighted cost = 9.0 | cut edges = 9
poisson_16_12_3 10 weighted cost = 7.0 | cut edges = 7
poisson_16_12_3 11 weighted cost = 7.0 | cut edges = 7
poisson_16_12_3 12 weighted cost = 7.0 | cut edges = 7
poisson_16_12_3 13 weighted cost = 10.0 | cut edges = 10
poisson_16_12_3 14 weighted cost = 7.0 | cut edges = 7
poisson_16_12_3 15 weighted cost = 8.0 | cut edges = 8
poisson_16_12_3 16 weighted cost = 8.0 | cut edges = 8
poisson_16_12_3 17 weighted cost = 5.0 | cut edges = 5
poisson_16_12_3 18 weighted cost = 7.0 | cut edges = 7
pois

In [8]:
import pandas as pd
poisson_df = pd.DataFrame(poisson_results)

poisson_df.to_csv(
    "poisson_exact_results.csv",
    index=False
)

In [13]:
import os
import glob
import numpy as np
import pandas as pd

folder = "data/congress-bills/numpy_20"
files = sorted(glob.glob(f"{folder}/*.npy"))

congress_results = []

for file in files:

    P = np.load(file)
    instance = os.path.splitext(os.path.basename(file))[0]

    n = P.shape[0]
    edges = get_edge_list(P)
    weights = np.ones(len(edges))


    x_opt, cut_opt, optimal_cost = (optimal_min_cut_balanced(n, edges,weights))

    congress_results.append({
        "dataset": "congress-bills",
        "instance": instance,
        "n_vertices": n,
        "n_edges": len(edges),
        "target_ones": (n + 1) // 2,
        "optimal_partition": "".join(map(str, x_opt)),
        "optimal_cost": optimal_cost,
        "cut_indicators": "".join(map(str, cut_opt))
    })

    print(
        instance,
        "n =", n,
        "m =", len(edges),
        "target ones =", (n + 1) // 2,
        "cost =", optimal_cost
    )

    print("partition:", x_opt)
    print("cut indicators:", cut_opt)
    print()

congress-bills_01 n = 2 m = 1 target ones = 1 cost = 1.0
partition: [1 0]
cut indicators: [1]

congress-bills_02 n = 4 m = 2 target ones = 2 cost = 0.0
partition: [1 0 1 0]
cut indicators: [0 0]

congress-bills_03 n = 5 m = 3 target ones = 3 cost = 0.0
partition: [1 0 0 1 1]
cut indicators: [0 0 0]

congress-bills_04 n = 10 m = 7 target ones = 5 cost = 1.0
partition: [0 1 0 0 0 1 1 1 0 1]
cut indicators: [0 0 0 1 0 0 0]

congress-bills_05 n = 9 m = 11 target ones = 5 cost = 0.0
partition: [0 1 0 1 0 1 0 1 1]
cut indicators: [0 0 0 0 0 0 0 0 0 0 0]

congress-bills_06 n = 14 m = 8 target ones = 7 cost = 0.0
partition: [1 1 1 0 0 0 0 0 0 1 1 1 0 1]
cut indicators: [0 0 0 0 0 0 0 0]

congress-bills_07 n = 18 m = 10 target ones = 9 cost = 0.0
partition: [0 1 0 1 1 1 0 1 1 1 0 1 0 0 0 0 0 1]
cut indicators: [0 0 0 0 0 0 0 0 0 0]

congress-bills_08 n = 4 m = 2 target ones = 2 cost = 0.0
partition: [1 1 0 0]
cut indicators: [0 0]

congress-bills_09 n = 14 m = 11 target ones = 7 cost = 2.0
part

In [14]:
# congress-bills_03 n = 5 m = 3 target ones = 3 cost = 0.0
# partition: [1 0 0 1 1]
# cut indicators: [0 0 0]

In [15]:

congress_df = pd.DataFrame(congress_results)

congress_df.to_csv(
    "congress_bills_exact_results.csv",
    index=False
)